# Vidya — M2 Screw Defect Training (Clean Start)
This is a fresh notebook. Run cells from top to bottom, one at a time. Do not use **Run all**.

**Dataset:** M2 screw defect detection, Mendeley Data v2, DOI `10.17632/2y6mcsyrpy.2`. Publisher lists 5,536 images in Body/Head OK/NG folders. Licence: CC BY 4.0. Source: https://data.mendeley.com/datasets/2y6mcsyrpy/2

This notebook trains two separate Body and Head binary classifiers using the dataset's provided Train/Test folders. It does not claim general screw/part recognition or repair capability. Training only counts as completed after the notebook reports real downloaded images and evaluation metrics.


## 1. Prepare the Colab workspace
Run the code cell below.

In [ ]:
from pathlib import Path
import os, sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests', 'tqdm', 'pillow', 'torch', 'torchvision'], check=True)
WORK = Path('/content/vidya_m2_clean')
RAW = WORK / 'archives'
EXTRACT = WORK / 'extracted'
OUT = WORK / 'outputs'
for p in (RAW, EXTRACT, OUT):
    p.mkdir(parents=True, exist_ok=True)
print('Workspace ready:', WORK)
import torch
print('GPU available:', torch.cuda.is_available())


## 2. Get the official public dataset file list
The earlier notebook used the authenticated API route and received HTTP 401. This notebook uses Mendeley's documented **public dataset files** route instead. If the public route is unavailable, stop and show the error; do not guess file IDs or download URLs.

In [ ]:
import requests, json

DATASET_ID = '2y6mcsyrpy'
VERSION = 2
API = 'https://api.data.mendeley.com'
headers = {
    'Accept': 'application/vnd.mendeley-public-dataset.1+json',
    'User-Agent': 'Mozilla/5.0 Vidya-Research-Notebook/1.0'
}

# IMPORTANT: Mendeley documents dataset metadata at /datasets/{id};
# the /datasets/publics/{id} route is an institution listing route,
# not the dataset metadata endpoint.
meta_url = f'{API}/datasets/{DATASET_ID}'
files_url = f'{API}/datasets/publics/{DATASET_ID}/files'

meta_r = requests.get(meta_url, params={'version': VERSION}, headers=headers, timeout=60)
print('Dataset metadata HTTP:', meta_r.status_code)
if meta_r.status_code == 401:
    print('Mendeley API requires authentication for metadata in this session. We will still test the documented public files endpoint.')
elif meta_r.status_code == 404:
    print('Dataset metadata was not found for this ID/version. Stop and check the publisher page.')
    meta_r.raise_for_status()
else:
    meta_r.raise_for_status()
    metadata = meta_r.json()
    print('Dataset:', metadata.get('name'), '| version:', metadata.get('version'))

# Official documented endpoint for a public dataset's files
files_r = requests.get(files_url, params={'version': VERSION, '$limit': 100}, headers=headers, timeout=60)
print('Public files HTTP:', files_r.status_code)
files_r.raise_for_status()
payload = files_r.json()
if isinstance(payload, dict):
    FILES = payload.get('files') or payload.get('results') or payload.get('items') or []
else:
    FILES = payload
assert FILES, 'The official public files endpoint returned no file records. Stop; do not guess URLs.'
for f in FILES:
    cd = f.get('content_details') or {}
    print(f.get('filename') or f.get('name'), '| bytes:', f.get('size') or cd.get('size'), '| id:', f.get('id'))

ARCHIVE_FILES = [f for f in FILES if (f.get('filename') or f.get('name') or '').lower().endswith(('.rar', '.zip', '.7z'))]
assert ARCHIVE_FILES, 'No archive records found in the official public file list.'
print('\nVerified archive candidates:', [f.get('filename') or f.get('name') for f in ARCHIVE_FILES])


## 3. Download the publisher's original archives
This downloads only file records returned by Mendeley. Keep the Colab tab open. The two archives are about 3.74 GB combined according to the publisher page, so download time depends on your connection and free Colab limits.

In [ ]:
import time
from tqdm.auto import tqdm

def download_file_record(item):
    name = item.get('filename') or item.get('name') or 'dataset_archive'
    file_id = item.get('id')
    if not file_id:
        raise RuntimeError(f'Mendeley record for {name} has no file ID; refusing to guess a URL.')
    dest = RAW / name
    if dest.exists() and dest.stat().st_size > 0:
        print('Already present:', name, dest.stat().st_size, 'bytes')
        return dest
    url = f'{API}/datasets/{DATASET_ID}/files/{file_id}/file_downloaded'
    with requests.get(url, params={'version': VERSION}, headers=headers, stream=True, allow_redirects=True, timeout=(30, 180)) as resp:
        print(name, 'download HTTP:', resp.status_code, '| final host:', requests.utils.urlparse(resp.url).netloc)
        resp.raise_for_status()
        total = int(resp.headers.get('content-length', 0))
        tmp = dest.with_suffix(dest.suffix + '.part')
        with open(tmp, 'wb') as out:
            bar = tqdm(total=total or None, unit='B', unit_scale=True, desc=name)
            for chunk in resp.iter_content(chunk_size=1024*1024):
                if chunk:
                    out.write(chunk)
                    bar.update(len(chunk))
            bar.close()
        if tmp.stat().st_size == 0:
            tmp.unlink(missing_ok=True)
            raise RuntimeError(f'{name} downloaded as an empty file.')
        tmp.rename(dest)
    print('Saved:', dest, '| bytes:', dest.stat().st_size)
    return dest

downloaded_archives = [download_file_record(item) for item in ARCHIVE_FILES]
print('\nDownloaded archive count:', len(downloaded_archives))
print('Total bytes:', sum(p.stat().st_size for p in downloaded_archives))


## 4. Extract and inspect the real image folders
The extraction cell checks the archive tool result. It does not create or fabricate images.

In [ ]:
import shutil
if shutil.which('7z') is None:
    subprocess.run(['apt-get','-qq','update'], check=True)
    subprocess.run(['apt-get','-qq','install','-y','p7zip-full'], check=True)

for archive in downloaded_archives:
    target = EXTRACT / archive.stem
    target.mkdir(parents=True, exist_ok=True)
    print('Extracting:', archive.name)
    result = subprocess.run(['7z','x','-y',f'-o{target}',str(archive)], text=True, capture_output=True)
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError(f'Extraction failed for {archive.name}; stop here.')

image_exts = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
all_images = [p for p in EXTRACT.rglob('*') if p.is_file() and p.suffix.lower() in image_exts]
print('\nActual extracted image files:', len(all_images))
for p in list(EXTRACT.rglob('*'))[:60]:
    if p.is_dir(): print('DIR ', p)
    elif p.is_file(): print('FILE', p.name)
assert all_images, 'No image files were found after extraction. Do not start training.'


## 5. Train Body and Head classifiers
Uses the dataset's own Train and Test folders (no synthetic images). If the expected folder layout is not found, the cell stops rather than silently making a misleading split.

In [ ]:
import torch, torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from pathlib import Path
import copy, json, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type != 'cuda': print('Warning: GPU not available; CPU training may be very slow.')

def find_component_root(component):
    wanted = component.lower()
    for d in EXTRACT.rglob('*'):
        if d.is_dir() and d.name.lower() == wanted:
            names = {x.name.lower() for x in d.iterdir() if x.is_dir()}
            if 'train' in names and 'test' in names:
                return d
    for d in EXTRACT.rglob('*'):
        if d.is_dir():
            names = {x.name.lower() for x in d.iterdir() if x.is_dir()}
            if 'train' in names and 'test' in names and wanted in str(d).lower():
                return d
    raise FileNotFoundError(f'Could not find {component}/Train and {component}/Test folders. Inspect extracted folders; do not guess labels.')

transform = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
results = {}
for component in ['Body','Head']:
    root = find_component_root(component)
    train_dir = next(p for p in root.iterdir() if p.is_dir() and p.name.lower() == 'train')
    test_dir = next(p for p in root.iterdir() if p.is_dir() and p.name.lower() == 'test')
    train_ds = datasets.ImageFolder(train_dir, transform=transform)
    test_ds = datasets.ImageFolder(test_dir, transform=transform)
    print(component, 'classes:', train_ds.classes, '| train:', len(train_ds), '| test:', len(test_ds))
    if len(train_ds.classes) != 2 or train_ds.classes != test_ds.classes:
        raise RuntimeError(f'{component}: expected matching two-class Train/Test folders.')
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)
    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    model = models.mobilenet_v3_small(weights=weights)
    model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, len(train_ds.classes))
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    criterion = torch.nn.CrossEntropyLoss()
    best_acc = -1.0
    epochs = 5
    for epoch in range(epochs):
        model.train(); running_loss=0.0; correct=0; count=0
        for xb,yb in train_loader:
            xb,yb=xb.to(device),yb.to(device); optimizer.zero_grad()
            logits=model(xb); loss=criterion(logits,yb); loss.backward(); optimizer.step()
            running_loss += loss.item()*yb.size(0); correct += (logits.argmax(1)==yb).sum().item(); count += yb.size(0)
        model.eval(); test_correct=0; test_count=0
        with torch.no_grad():
            for xb,yb in test_loader:
                xb,yb=xb.to(device),yb.to(device); pred=model(xb).argmax(1)
                test_correct += (pred==yb).sum().item(); test_count += yb.size(0)
        test_acc=test_correct/max(test_count,1)
        print(f'{component} epoch {epoch+1}/{epochs} loss={running_loss/max(count,1):.4f} train_acc={correct/max(count,1):.4f} test_acc={test_acc:.4f}')
        if test_acc > best_acc:
            best_acc=test_acc
            torch.save({'state_dict':model.state_dict(),'classes':train_ds.classes,'component':component}, OUT/f'{component.lower()}_mobilenetv3_small.pth')
    results[component]={'train_images':len(train_ds),'test_images':len(test_ds),'classes':train_ds.classes,'best_test_accuracy':best_acc}

report = {'dataset_id':DATASET_ID,'version':VERSION,'device':str(device),'results':results,'note':'Metrics are from this actual run; they are not guaranteed to generalize beyond this dataset.'}
(OUT/'training_report.json').write_text(json.dumps(report,indent=2))
print('\nTRAINING RUN FINISHED')
print(json.dumps(report,indent=2))


## 6. Package the trained outputs
Run only after the training cell finishes successfully. This creates a ZIP containing the model weights and actual training report.

In [ ]:
import zipfile
report_path = OUT / 'training_report.json'
assert report_path.exists(), 'Training report not found. Training has not completed successfully.'
zip_path = WORK / 'vidya_m2_trained_outputs.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file(): z.write(p, arcname=p.name)
print('Package:', zip_path)
print('Package size:', zip_path.stat().st_size, 'bytes')
print('Download from Colab file browser or run:')
print('from google.colab import files; files.download(str(zip_path))')


## Dataset attribution and limitations
Prastio, Rizki Putra; Indrawan, Rodik Wahyu (2026), “M2 screw defect detection dataset”, Mendeley Data, V2, DOI: 10.17632/2y6mcsyrpy.2. Licence: CC BY 4.0.

This is a narrow binary defect-classification experiment for M2 screw Body and Head images. It is not evidence of universal screw recognition, part compatibility, safe repair instructions, or on-device Vidya performance. Record actual downloaded image counts, runtime, hardware, and metrics from completed execution only.
